# FlashAttention/IO-aware Attention 실습 - 추가 실습 코드: Attention의 메모리 접근 분석

- Tutorial ID: `mod-flashattention-lab`
- Tutorial: FlashAttention/IO-aware Attention 실습
- Section ID: `practice-io-complexity`
- Section: 추가 실습 코드: Attention의 메모리 접근 분석


In [ ]:
# ============================================================
# 코드 읽는 법 — 추가 실습 코드: Attention의 메모리 접근 분석
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

def analyze_memory_access(seq_len, d_model, sram_size_mb=20):
    """Standard vs FlashAttention 메모리 접근 분석"""
    bytes_per_float = 4  # FP32
    
    # Standard Attention: N^2 메모리
    standard_qkv = 3 * seq_len * d_model * bytes_per_float
    standard_attention_matrix = seq_len * seq_len * bytes_per_float * 2
        standard_output = seq_len * d_model * bytes_per_float
    standard_total = standard_qkv + standard_attention_matrix + standard_output
    
    # FlashAttention: 블록 단위 처리
    sram_bytes = sram_size_mb * 1024 * 1024
    block_size = int(np.sqrt(sram_bytes / bytes_per_float / 4))
    num_blocks = (seq_len + block_size - 1) // block_size
    flash_per_block = 4 * block_size * d_model * bytes_per_float
    flash_total = num_blocks * num_blocks * flash_per_block
    
    return {
        'seq_len': seq_len,
        'standard_mb': standard_total / 1e6,
        'flash_mb': flash_total / 1e6,
        'attn_matrix_mb': standard_attention_matrix / 1e6,
        'speedup': standard_total / flash_total if flash_total > 0 else 0
    }

print("=" * 60)
print("FlashAttention Memory Analysis")
print("=" * 60)

print("\\n시퀀스 길이별 메모리 사용량 비교:\\n")
print(f"{'Seq Len':>8} | {'Standard':>10} | {'Flash':>10} | {'Attn Mat':>10} | {'Speedup':>8}")
print("-" * 60)

for seq_len in [512, 1024, 2048, 4096, 8192, 16384]:
        r = analyze_memory_access(seq_len, d_model=64)
    bar = "█" * int(min(r['speedup'], 10))
    print(f"{r['seq_len']:>8} | {r['standard_mb']:>8.1f}MB | {r['flash_mb']:>8.1f}MB | {r['attn_matrix_mb']:>8.1f}MB | {r['speedup']:>6.2f}x {bar}")

print("\\n💡 핵심 통찰:")
print("  - Attention Matrix (N²)가 메모리 병목의 주범")
print("  - FlashAttention은 이를 저장하지 않고 on-the-fly 계산")
print("  - 긴 시퀀스에서 효과 극대화")